# Assignment 2
**CS5131 / CS6031 Computer Vision** &nbsp;·&nbsp; Due: **Tuesday, October 6, 2026**, 11:59 PM &nbsp;·&nbsp; 65 points

In this assignment you will derive the gradients of a neural network by hand, implement
backpropagation for a multilayer perceptron, and train it on MNIST.

| Part | Topic | Points |
|---|---|---|
| **A** | Theory: three derivations | 30 |
| **B** | Backpropagation for an MLP | 35 |

**Reading.** Chapters 12 (Neural Networks) and 14 (Backpropagation) of *Foundations of
Computer Vision*, and §24.2 (Convolutional Layers) for Question A2(b).

**Rules for the code.**
- Use **NumPy only**. No PyTorch, TensorFlow, JAX, autograd libraries or anything that
  computes gradients for you.
- Edit only the cells marked `# YOUR CODE HERE` and the *Your answer* cells.
- Every layer comes with a **gradient check** cell. Get it to print `PASS` before moving on:
  a layer that fails its check will make every later part fail in confusing ways.


**AI tools.** Do not use the AI tools to give you the answers. You may end up getting the points for this assignment, but it does not increase your mastery on the topic in any way. You can use AI tools to understand the concepts. In this case, please leave a note as to where you used AI.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from a2utils import load_mnist, batches, evaluate, grad_check, grad_check_model

np.set_printoptions(precision=4, suppress=True)

---
# Part A — Theory &nbsp; [30 points]

Answer in the markdown cells. Show your working.

## A1. Softmax and cross-entropy &nbsp; [14 points]

A classifier produces logits $\mathbf{z} \in \mathbb{R}^K$. The softmax turns them into
probabilities, and the cross-entropy compares these with a one-hot target $\mathbf{y}$:

$$\hat{y}_j = \frac{e^{z_j}}{\sum_{l=1}^{K} e^{z_l}}, \qquad J = -\sum_{j=1}^{K} y_j \log \hat{y}_j .$$

**(a) [6]** Show that
$$\frac{\partial \hat{y}_j}{\partial z_k} = \hat{y}_j\,(\delta_{jk} - \hat{y}_k),$$
where $\delta_{jk}$ is 1 if $j = k$ and 0 otherwise. Treat the cases $j = k$ and $j \neq k$ separately.

**(b) [6]** Use (a) and the chain rule to show that
$$\frac{\partial J}{\partial \mathbf{z}} = \hat{\mathbf{y}} - \mathbf{y}.$$
State clearly where you use the fact that $\mathbf{y}$ is one-hot. Does the result still hold if $\mathbf{y}$ is any probability vector, not necessarily one-hot?

**(c) [2]** In practice the loss is averaged over a batch of $B$ examples,
$J_{\text{batch}} = \frac{1}{B}\sum_{b=1}^{B} J^{(b)}$. What is $\partial J_{\text{batch}} / \partial \mathbf{z}^{(b)}$
for one example $b$ in the batch?

*You will implement exactly this gradient in B3.*

### Your answer — A1

*Write your answer here. Use LaTeX for mathematics, e.g. `$\frac{\partial J}{\partial z}$`.*

## A2. Shared parameters &nbsp; [8 points]

**(a) [4]** A scalar parameter $\theta$ is used in two places in a computation graph, so that
$J = f\big(g(\theta),\, h(\theta)\big)$. Show that
$$\frac{dJ}{d\theta} = \frac{\partial f}{\partial g}\frac{dg}{d\theta} + \frac{\partial f}{\partial h}\frac{dh}{d\theta}.$$
Explain how this relates to the **branch** module of Chapter 14, §14.7.

**(b) [4]** A one-dimensional convolutional layer computes
$x_{\text{out}}[n] = \sum_{k} w[k]\, x_{\text{in}}[n+k]$. The same weight $w[k]$ is used at every
position $n$. Using (a), write $\partial J / \partial w[k]$ in terms of the incoming gradient
$g[n] = \partial J / \partial x_{\text{out}}[n]$ and the input. Then explain in two or three
sentences why this rule is what makes convolutional layers trainable, and why it lets them
learn from less data than a fully connected layer.

*You will implement the two-dimensional version of this in the next assignment.*

### Your answer — A2

*Write your answer here. Use LaTeX for mathematics, e.g. `$\frac{\partial J}{\partial z}$`.*

## A3. Numerical stability of the softmax &nbsp; [8 points]

**(a) [3]** Show that $\text{softmax}(\mathbf{z}) = \text{softmax}(\mathbf{z} - c)$ for any scalar constant $c$.

**(b) [3]** Run the cell below, which computes the softmax of $\mathbf{z} = [1000,\, 0,\, -1000]$
directly from the definition. Explain what goes wrong and why. Then explain what goes wrong for
$\mathbf{z} = [-1000,\, -1001,\, -1002]$, which is a different failure.

**(c) [2]** Stable implementations use $c = \max_j z_j$. Why the maximum specifically, rather than
some other constant? And why do implementations of cross-entropy usually compute
$\log \hat{y}$ directly, as $z_j - c - \log\sum_l e^{z_l - c}$, rather than computing $\hat{y}$ first and
then taking its log?

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for z in (np.array([1000., 0., -1000.]), np.array([-1000., -1001., -1002.])):
        e = np.exp(z)
        print("z =", z, "\n   exp(z) =", e, "\n   naive softmax =", e / e.sum(), "\n")

### Your answer — A3

*Write your answer here. Use LaTeX for mathematics, e.g. `$\frac{\partial J}{\partial z}$`.*

---
# Part B — Backpropagation for an MLP &nbsp; [35 points]

Each layer is a class with a `forward` method, a `backward` method and a `params` method,
following Figure 14.11. The one difference from the book is that we process a **batch** of
datapoints at once, stored as the **rows** of a matrix:

| | Chapter 14 | this assignment |
|---|---|---|
| one datapoint | column vector $\mathbf{x} \in \mathbb{R}^{N}$ | row of $X \in \mathbb{R}^{B \times N}$ |
| linear layer | $\mathbf{x}_{\text{out}} = W\mathbf{x}_{\text{in}} + \mathbf{b}$ | `X_out = X_in @ W.T + b` |
| gradient | row vector $\mathbf{g} \in \mathbb{R}^{1\times M}$ | `G`, shape `(B, M)`, same as `X_out` |

**The contract every layer follows.**
- `forward(x)` computes the output and **saves whatever `backward` will need** on `self`.
- `backward(g_out)` receives $\partial J/\partial \mathbf{x}_{\text{out}}$ (same shape as the output),
  stores the parameter gradients in `self.dW`, `self.db`, and **returns**
  $\partial J/\partial \mathbf{x}_{\text{in}}$ (same shape as the input).
- `params()` returns a list of `(name, parameter, gradient)` triples.

## B1. Linear layer &nbsp; [8 points]

Implement `Linear.backward`. It must set `self.dW` and `self.db`, and return the gradient
with respect to the input.

*Hints.* Check your shapes before anything else: `self.dW` must have the same shape as
`self.W`, and the value you return must have the same shape as `self.x`. The bias is added to
every row of the batch, so its gradient collects a contribution from every row.

In [ ]:
class Linear:
    """
    x_out = x_in @ W.T + b
      x_in : (B, n_in)     W : (n_out, n_in)     b : (n_out,)     x_out : (B, n_out)
    """
    def __init__(self, n_in, n_out, rng):
        self.W = rng.standard_normal((n_out, n_in)) * np.sqrt(2.0 / n_in)   # He initialization
        self.b = np.zeros(n_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, x):
        self.x = x                        # save the operating point for backward
        return x @ self.W.T + self.b

    def backward(self, g_out):
        """g_out: (B, n_out).  Set self.dW and self.db; return dJ/dx_in of shape (B, n_in)."""
        # YOUR CODE HERE
        raise NotImplementedError

    def params(self):
        return [("W", self.W, self.dW), ("b", self.b, self.db)]

In [ ]:
rng = np.random.default_rng(1)
print("Linear:")
ok_B1 = grad_check(Linear(6, 4, rng), rng.standard_normal((5, 6)))

## B2. ReLU &nbsp; [3 points]

Implement `ReLU.forward` and `ReLU.backward`. The backward pass should act as the **gate**
described in Chapter 14, §14.6.2.

In [ ]:
class ReLU:
    def forward(self, x):
        # YOUR CODE HERE
        raise NotImplementedError

    def backward(self, g_out):
        # YOUR CODE HERE
        raise NotImplementedError

    def params(self):
        return []

In [ ]:
rng = np.random.default_rng(2)
print("ReLU:")
ok_B2 = grad_check(ReLU(), rng.standard_normal((5, 6)))

## B3. Softmax with cross-entropy &nbsp; [8 points]

Implement both methods. `forward(z, y)` takes logits `z` of shape `(B, K)` and **integer**
labels `y` of shape `(B,)`, and returns the **mean** cross-entropy over the batch as a float.
`backward()` takes no argument, because this is the last layer, and returns
$\partial J / \partial \mathbf{z}$ of shape `(B, K)`.

Your forward pass **must be numerically stable** (Question A3), and your backward pass should use
the result of Question A1. The test below checks both.

*Hint.* `y` holds integers, not one-hot vectors. `z[np.arange(B), y]` picks out the entry of
each row belonging to the true class.

In [ ]:
class SoftmaxCrossEntropy:
    def forward(self, z, y):
        """z: (B, K) logits.  y: (B,) integer labels.  Returns the mean loss (a float)."""
        # YOUR CODE HERE
        raise NotImplementedError

    def backward(self):
        """Returns dJ/dz, shape (B, K)."""
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
rng = np.random.default_rng(3)
sce = SoftmaxCrossEntropy()
z, y = rng.standard_normal((5, 7)), rng.integers(0, 7, 5)
sce.forward(z, y); analytic = sce.backward()
numeric, eps = np.zeros_like(z), 1e-6
for i in range(z.shape[0]):
    for j in range(z.shape[1]):
        zp, zm = z.copy(), z.copy(); zp[i, j] += eps; zm[i, j] -= eps
        numeric[i, j] = (sce.forward(zp, y) - sce.forward(zm, y)) / (2 * eps)
rel = np.abs(analytic - numeric).max() / np.abs(analytic + numeric).max()
print(f"SoftmaxCrossEntropy gradient   relative error {rel:.2e}   {'PASS' if rel < 1e-5 else 'FAIL'}")
big = sce.forward(np.array([[1000., 0., -1000.], [-1000., -1001., -1002.]]), np.array([0, 2]))
print(f"Stability on extreme logits    loss = {big:.4f}   {'PASS' if np.isfinite(big) else 'FAIL'}")

## B4. The network &nbsp; [4 points]

`MLP` stacks linear layers with ReLUs between them. Implement `MLP.backward`, which runs the
backward pass through every layer in reverse order. The rest of the class is given.

In [ ]:
class MLP:
    def __init__(self, sizes, rng):
        self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Linear(sizes[i], sizes[i + 1], rng))
            if i < len(sizes) - 2:
                self.layers.append(ReLU())
        self.loss = SoftmaxCrossEntropy()

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, g):
        """g is dJ/d(output of the last layer). Propagate it back through every layer."""
        # YOUR CODE HERE
        raise NotImplementedError

    def params(self):
        out = []
        for layer in self.layers:
            out += layer.params()
        return out

    def step(self, lr):
        """One step of gradient descent on every parameter."""
        for name, p, g in self.params():
            p -= lr * g

In [ ]:
rng = np.random.default_rng(4)
print("Whole MLP, loss included:")
ok_B4 = grad_check_model(MLP([10, 8, 6, 4], rng), rng.standard_normal((5, 10)), rng.integers(0, 4, 5))

## B5. Training on MNIST &nbsp; [12 points]

**(a) [4]** Complete the inner loop of `train`: a forward pass, the loss, a backward pass, and a
gradient-descent step. `train` is written so that it will also work unchanged for the CNN in the next assignment.

**(b) [4]** Run the cell after it. Train `MLP([784, 128, 64, 10])` for 3 epochs. A correct
implementation reaches **at least 95%** test accuracy, and should take well under a minute.

**(c) [4]** In the answer cell: report your final test accuracy, include the loss curve, and
describe its shape. Why is the per-batch loss so noisy, and what would you change to make the curve
smoother without changing what the network learns?

In [3]:
def train(net, x_train, y_train, x_test, y_test, epochs, lr, batch_size, rng):
    """Returns a list of per-batch training losses."""
    history = []
    for epoch in range(epochs):
        for xb, yb in batches(x_train, y_train, batch_size, rng):
            # YOUR CODE HERE: forward pass, loss, backward pass, step.
            #                 Append the loss (a float) to history.
            raise NotImplementedError
        print(f"epoch {epoch + 1}:  last batch loss {history[-1]:.4f},  "
              f"test accuracy {evaluate(net, x_test, y_test):.4f}")
    return history

In [4]:
x_train, y_train, x_test, y_test = load_mnist()
print("train", x_train.shape, " test", x_test.shape)

rng = np.random.default_rng(0)
mlp = MLP([784, 128, 64, 10], rng)
hist_mlp = train(mlp, x_train, y_train, x_test, y_test, epochs=3, lr=0.1, batch_size=64, rng=rng)

plt.figure(figsize=(8, 3))
plt.plot(hist_mlp, lw=0.5)
plt.xlabel("batch"); plt.ylabel("training loss"); plt.title("MLP"); plt.yscale("log")
plt.show()
acc_mlp = evaluate(mlp, x_test, y_test)
print(f"final MLP test accuracy: {acc_mlp:.4f}")

  downloading train-images-idx3-ubyte.gz from raw.githubusercontent.com ...
  downloading train-images-idx3-ubyte.gz from storage.googleapis.com ...
  downloading train-images-idx3-ubyte.gz from ossci-datasets.s3.amazonaws.com ...
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3442, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/w_/30b6d1vx1klb40qkjzph1l4c0000gp/T/ipykernel_29692/2348086123.py", line 1, in <module>
    x_train, y_train, x_test, y_test = load_mnist(flat=True)
  File "/Users/vikram/Downloads/a3utils.py", line 59, in load_mnist
    and put them, still compressed, in this folder:
  File "/Users/vikram/Downloads/a3utils.py", line 59, in <dictcomp>
    and put them, still compressed, in this folder:
  File "/Users/vikram/Downloads/a3utils.py", line 42, in _fetch
RuntimeError: could not download train-images-idx3-ubyte.gz from any mirror: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:997)>

During handling of the above exception, another exception occurred:

Traceback (most 

### Your answer — B5

*Write your answer here. Use LaTeX for mathematics, e.g. `$\frac{\partial J}{\partial z}$`.*

---
## Submission

Submit this notebook, **executed from top to bottom** with all outputs visible
(*Kernel → Restart & Run All*), together with `a2utils.py`. Every gradient check should print `PASS`.

**AI tool disclosure** (if applicable):